# Fingerprint Calculation

In [ ]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, rdMolDescriptors

# ======================================================
# 1. Fungsi Fingerprints: ECFP4 (Morgan), MACCS, APF
# ======================================================
def compute_ecfp4(smiles, n_bits=1024):
    mol = Chem.MolFromSmiles(str(smiles)) if isinstance(smiles, str) else None
    if mol:
        return list(AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=n_bits))
    return [0] * n_bits

def compute_maccs_fp(smiles):
    mol = Chem.MolFromSmiles(str(smiles)) if isinstance(smiles, str) else None
    if mol:
        return list(MACCSkeys.GenMACCSKeys(mol))
    return [0] * 167  # panjang MACCS = 167

def compute_apf_fp(smiles, n_bits=1024):
    mol = Chem.MolFromSmiles(str(smiles)) if isinstance(smiles, str) else None
    if mol:
        return list(rdMolDescriptors.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=n_bits))
    return [0] * n_bits

# ======================================================
# 2. Input files: Dev + ExternalVal1/2/3 untuk 3 ion channel
# ======================================================
input_files = [
    r"D:\QSAR Cardiotoxicity\Dataset Splitting\Cav1.2\Cav1.2_ExternalVal3_hard.csv",
    r"D:\QSAR Cardiotoxicity\Dataset Splitting\Cav1.2\Cav1.2_DevSet.csv",
    r"D:\QSAR Cardiotoxicity\Dataset Splitting\Cav1.2\Cav1.2_ExternalVal1_easy.csv",
    r"D:\QSAR Cardiotoxicity\Dataset Splitting\Cav1.2\Cav1.2_ExternalVal2_medium.csv",

    r"D:\QSAR Cardiotoxicity\Dataset Splitting\hERG\hERG_ExternalVal3_hard.csv",
    r"D:\QSAR Cardiotoxicity\Dataset Splitting\hERG\hERG_DevSet.csv",
    r"D:\QSAR Cardiotoxicity\Dataset Splitting\hERG\hERG_ExternalVal1_easy.csv",
    r"D:\QSAR Cardiotoxicity\Dataset Splitting\hERG\hERG_ExternalVal2_medium.csv",

    r"D:\QSAR Cardiotoxicity\Dataset Splitting\Nav1.5\Nav1.5_ExternalVal3_hard.csv",
    r"D:\QSAR Cardiotoxicity\Dataset Splitting\Nav1.5\Nav1.5_DevSet.csv",
    r"D:\QSAR Cardiotoxicity\Dataset Splitting\Nav1.5\Nav1.5_ExternalVal1_easy.csv",
    r"D:\QSAR Cardiotoxicity\Dataset Splitting\Nav1.5\Nav1.5_ExternalVal2_medium.csv",
]

# folder output global
out_root = r"D:\QSAR Cardiotoxicity\Descriptor Calculation\Fingerprint"
os.makedirs(out_root, exist_ok=True)

# ======================================================
# 3. Loop utama
# ======================================================
for input_path in input_files:

    print(f"\n🔄 Processing: {input_path}")
    if not os.path.exists(input_path):
        print(f"   [WARNING] File not found, skipped.")
        continue

    df = pd.read_csv(input_path)
    assert "SMILES" in df.columns, "Kolom 'SMILES' tidak ditemukan."

    # Fingerprints: ECFP4, MACCS, APF
    df["ECFP4"] = df["SMILES"].apply(compute_ecfp4)
    df["MACCS"] = df["SMILES"].apply(compute_maccs_fp)
    df["APF"]   = df["SMILES"].apply(compute_apf_fp)

    # buat nama file output berdasarkan nama file input
    base_name = os.path.basename(input_path).replace(".csv", "_with_ECFP4_MACCS_APF.csv")
    output_path = os.path.join(out_root, base_name)

    df.to_csv(output_path, index=False)
    print(f"✅ Saved to {output_path}")

print("\n🎉 All split files processed successfully!")


# Graph Calculation

In [ ]:
import os
import pandas as pd
import pickle
from rdkit import Chem
from rdkit.Chem import rdchem
from joblib import Parallel, delayed  # pip install joblib

# ======================================================
# 0. Pengaturan jumlah core
# ======================================================
N_JOBS = 6

# ======================================================
# 1. Atom feature: 67 dimensi
# ======================================================

ATOM_LIST = [
    'C','N','O','F','Cl','S','Br','P','I','Na','H','Si','Sn','Hg','B','Ca','Se','K',
    'Zn','Bi','Cd','Cr','Li','In','Sb','As','Fe','Cu','Pb','Mg','Al','V','Tl','Ag',
    'Pd','Co','Ti','Ge','Au','Ni','Mn','Pt'
]

def atom_to_feature_vector(atom: rdchem.Atom):
    # 1) Atom symbol (42)
    symbol = atom.GetSymbol()
    atom_type = [int(symbol == a) for a in ATOM_LIST]

    # 2) Degree 0–6 (7)
    degree = atom.GetDegree()
    if degree > 6:
        degree = 6
    degree_enc = [int(degree == i) for i in range(7)]

    # 3) Number of connected hydrogens 0–4 (5)
    num_h = atom.GetTotalNumHs()
    if num_h > 4:
        num_h = 4
    num_h_enc = [int(num_h == i) for i in range(5)]

    # 4) Implicit valence 0–5 (6)
    imp_val = atom.GetImplicitValence()
    if imp_val > 5:
        imp_val = 5
    imp_val_enc = [int(imp_val == i) for i in range(6)]

    # 5) In rings: ring size 3–8 (6)
    ring_sizes = [int(atom.IsInRingSize(size)) for size in range(3, 9)]

    # 6) In aromatic ring (1)
    aromatic = [int(atom.GetIsAromatic())]

    # Total 67
    return atom_type + degree_enc + num_h_enc + imp_val_enc + ring_sizes + aromatic

def mol_to_graph(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    node_features = [atom_to_feature_vector(atom) for atom in mol.GetAtoms()]

    edges = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edges.append([i, j])
        edges.append([j, i])

    return {
        "node_features": node_features,  # M x 67
        "edge_index": edges              # 2N x 2
    }

# ======================================================
# 2. Daftar file per ion channel & split
# ======================================================

file_map = {
    "Cav1.2": {
        "DevSet":              r"D:\QSAR Cardiotoxicity\Dataset Splitting\Cav1.2\Cav1.2_DevSet.csv",
        "ExternalVal1_easy":   r"D:\QSAR Cardiotoxicity\Dataset Splitting\Cav1.2\Cav1.2_ExternalVal1_easy.csv",
        "ExternalVal2_medium": r"D:\QSAR Cardiotoxicity\Dataset Splitting\Cav1.2\Cav1.2_ExternalVal2_medium.csv",
        "ExternalVal3_hard":   r"D:\QSAR Cardiotoxicity\Dataset Splitting\Cav1.2\Cav1.2_ExternalVal3_hard.csv",
    },
    "hERG": {
        "DevSet":              r"D:\QSAR Cardiotoxicity\Dataset Splitting\hERG\hERG_DevSet.csv",
        "ExternalVal1_easy":   r"D:\QSAR Cardiotoxicity\Dataset Splitting\hERG\hERG_ExternalVal1_easy.csv",
        "ExternalVal2_medium": r"D:\QSAR Cardiotoxicity\Dataset Splitting\hERG\hERG_ExternalVal2_medium.csv",
        "ExternalVal3_hard":   r"D:\QSAR Cardiotoxicity\Dataset Splitting\hERG\hERG_ExternalVal3_hard.csv",
    },
    "Nav1.5": {
        "DevSet":              r"D:\QSAR Cardiotoxicity\Dataset Splitting\Nav1.5\Nav1.5_DevSet.csv",
        "ExternalVal1_easy":   r"D:\QSAR Cardiotoxicity\Dataset Splitting\Nav1.5\Nav1.5_ExternalVal1_easy.csv",
        "ExternalVal2_medium": r"D:\QSAR Cardiotoxicity\Dataset Splitting\Nav1.5\Nav1.5_ExternalVal2_medium.csv",
        "ExternalVal3_hard":   r"D:\QSAR Cardiotoxicity\Dataset Splitting\Nav1.5\Nav1.5_ExternalVal3_hard.csv",
    }
}

# folder output global
out_root = r"D:\QSAR Cardiotoxicity\Descriptor Calculation\Graph"
os.makedirs(out_root, exist_ok=True)

# ======================================================
# 3. Loop utama: simpan (graph, Activity, SMILES) + preview top 10
# ======================================================

for target, splits in file_map.items():
    for split_name, input_path in splits.items():
        print(f"\n🔄 {target} - {split_name}")
        if not os.path.exists(input_path):
            print(f"[PERINGATAN] File tidak ditemukan: {input_path}")
            continue

        df = pd.read_csv(input_path)
        assert "SMILES" in df.columns, f"Kolom 'SMILES' tidak ditemukan di {input_path}"
        assert "Activity" in df.columns, f"Kolom 'Activity' (label) tidak ditemukan di {input_path}"

        smiles_list = df["SMILES"].astype(str).tolist()
        labels_list = df["Activity"].tolist()
        n_total = len(smiles_list)

        # Proses graph secara paralel
        graphs = Parallel(n_jobs=N_JOBS, verbose=1)(
            delayed(mol_to_graph)(smi) for smi in smiles_list
        )

        data_list = []
        n_invalid = 0
        for smi, lbl, g in zip(smiles_list, labels_list, graphs):
            if g is None:
                n_invalid += 1
                continue
            try:
                y = int(lbl)
            except Exception:
                y = lbl  # jika sudah dalam format yang diinginkan (misal float/bool)

            data_list.append(
                {
                    "graph": g,        # dict: node_features, edge_index
                    "Activity": y,     # label
                    "SMILES": smi      # untuk trace/debug
                }
            )

        out_name = f"{target}_{split_name}_graphs_67feat_with_labels.pkl"
        out_path = os.path.join(out_root, out_name)

        with open(out_path, "wb") as f:
            pickle.dump(data_list, f)

        print(f"  Total molekul input : {n_total}")
        print(f"  Invalid SMILES      : {n_invalid}")
        print(f"  Objek tersimpan     : {len(data_list)}")
        print(f"  ✅ Saved: {out_path}")

        # === Preview metadata TOP 10 isi pkl ===
        print(f"\n  --- Preview isi (max 10) untuk {out_name} ---")
        for i, item in enumerate(data_list[:10]):
            g = item["graph"]
            print(f"   [{i}] SMILES : {item['SMILES']}")
            print(f"       Activity: {item['Activity']}")
            print(f"       n_nodes : {len(g['node_features'])}, n_edges: {len(g['edge_index'])}")
        print("  --- End preview ---\n")

print("\n🎉 Graph + label (Activity) + SMILES selesai untuk semua kanal dan split.")


# Mordred 2D 3D

In [ ]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from mordred import Calculator, descriptors  # pip install "mordred[full]"

# =========================
# 1. Konfigurasi input & output
# =========================

file_map = {
    "Cav1.2": {
        "DevSet":              r"D:\QSAR Cardiotoxicity\Dataset Splitting\Cav1.2\Cav1.2_DevSet.csv",
        "ExternalVal1_easy":   r"D:\QSAR Cardiotoxicity\Dataset Splitting\Cav1.2\Cav1.2_ExternalVal1_easy.csv",
        "ExternalVal2_medium": r"D:\QSAR Cardiotoxicity\Dataset Splitting\Cav1.2\Cav1.2_ExternalVal2_medium.csv",
        "ExternalVal3_hard":   r"D:\QSAR Cardiotoxicity\Dataset Splitting\Cav1.2\Cav1.2_ExternalVal3_hard.csv",
    },
    "hERG": {
        "DevSet":              r"D:\QSAR Cardiotoxicity\Dataset Splitting\hERG\hERG_DevSet.csv",
        "ExternalVal1_easy":   r"D:\QSAR Cardiotoxicity\Dataset Splitting\hERG\hERG_ExternalVal1_easy.csv",
        "ExternalVal2_medium": r"D:\QSAR Cardiotoxicity\Dataset Splitting\hERG\hERG_ExternalVal2_medium.csv",
        "ExternalVal3_hard":   r"D:\QSAR Cardiotoxicity\Dataset Splitting\hERG\hERG_ExternalVal3_hard.csv",
    },
    "Nav1.5": {
        "DevSet":              r"D:\QSAR Cardiotoxicity\Dataset Splitting\Nav1.5\Nav1.5_DevSet.csv",
        "ExternalVal1_easy":   r"D:\QSAR Cardiotoxicity\Dataset Splitting\Nav1.5\Nav1.5_ExternalVal1_easy.csv",
        "ExternalVal2_medium": r"D:\QSAR Cardiotoxicity\Dataset Splitting\Nav1.5\Nav1.5_ExternalVal2_medium.csv",
        "ExternalVal3_hard":   r"D:\QSAR Cardiotoxicity\Dataset Splitting\Nav1.5\Nav1.5_ExternalVal3_hard.csv",
    }
}

out_root = r"D:\QSAR Cardiotoxicity\Descriptor Calculation\Mordred_3D"
os.makedirs(out_root, exist_ok=True)

# =========================
# 2. Utility: build 3D mol from SMILES
# =========================

def mol_from_smiles_3d(smi: str):
    """
    SMILES -> RDKit Mol with 3D coordinates.
    Return None if embedding/optimization fails.
    """
    if not isinstance(smi, str):
        return None
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    try:
        mol = Chem.AddHs(mol)
        # Embed 3D conformer
        res = AllChem.EmbedMolecule(mol, randomSeed=42)
        if res != 0:
            return None
        # UFF optimize
        res_opt = AllChem.UFFOptimizeMolecule(mol, maxIters=200)
        # Buang H eksplisit kalau tidak ingin (opsional)
        mol = Chem.RemoveHs(mol)
        return mol
    except Exception:
        return None

# =========================
# 3. Setup Mordred Calculator (2D+3D)
# =========================
calc = Calculator(descriptors, ignore_3D=False)

# =========================
# 4. Loop per file: hitung Mordred + simpan SMILES & Activity
# =========================

for target, splits in file_map.items():
    for split_name, input_path in splits.items():
        print(f"\n🔄 {target} - {split_name}")
        if not os.path.exists(input_path):
            print(f"[PERINGATAN] File tidak ditemukan: {input_path}")
            continue

        df = pd.read_csv(input_path)
        assert "SMILES" in df.columns,   f"Kolom 'SMILES' tidak ditemukan di {input_path}"
        assert "Activity" in df.columns, f"Kolom 'Activity' tidak ditemukan di {input_path}"

        smiles_list = df["SMILES"].astype(str).tolist()

        # Bangun Mol 3D per SMILES
        mols_3d = []
        valid_idx = []
        n_invalid = 0
        for i, smi in enumerate(smiles_list):
            m = mol_from_smiles_3d(smi)
            if m is None:
                n_invalid += 1
            else:
                mols_3d.append(m)
                valid_idx.append(i)

        if len(mols_3d) == 0:
            print("  [PERINGATAN] Tidak ada molekul 3D valid, skip file ini.")
            continue

        print(f"  Molekul input  : {len(smiles_list)}")
        print(f"  Molekul 3D ok  : {len(mols_3d)}")
        print(f"  Invalid (3D)   : {n_invalid}")

        # Subset df ke hanya yang 3D-nya sukses
        df_valid = df.iloc[valid_idx].reset_index(drop=True)

        # Hitung Mordred di mols_3d
        df_desc = calc.pandas(mols_3d)
        df_desc = df_desc.dropna(axis=1, how="all").fillna(0)

        # Gabungkan dengan SMILES + Activity (dan kolom lain jika ingin)
        base_cols = ["SMILES", "Activity"]
        other_cols = [c for c in df_valid.columns if c not in base_cols]
        df_base = df_valid[base_cols + other_cols].reset_index(drop=True)

        df_out = pd.concat([df_base, df_desc.reset_index(drop=True)], axis=1)

        # Simpan
        out_name = f"{target}_{split_name}_Mordred_2D3D_from3Dmol.csv"
        out_path = os.path.join(out_root, out_name)
        df_out.to_csv(out_path, index=False)

        print(f"  Molekul tersimpan   : {len(df_out)}")
        print(f"  Kolom descriptor    : {df_desc.shape[1]}")
        print(f"  ✅ Saved: {out_path}")

        # Preview metadata top 3 baris
        print("  --- Preview top 3 ---")
        preview_cols = ["SMILES", "Activity"] + list(df_desc.columns[:5])
        print(df_out[preview_cols].head(3).to_string(index=False))
        print("  --- End preview ---")

print("\n🎉 Perhitungan Mordred 2D+3D (dengan RDKit 3D embedding) selesai untuk semua kanal dan split.")


In [ ]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from mordred import Calculator, descriptors  # pip install "mordred[full]"

# =========================
# 1. Konfigurasi input & output - HANYA hERG DevSet
# =========================

file_path = r"D:\\QSAR Cardiotoxicity\\Dataset Splitting\\hERG\\hERG_DevSet.csv"
out_root = r"D:\\QSAR Cardiotoxicity\\Descriptor Calculation\\Mordred_3D"
out_path = os.path.join(out_root, "hERG_DevSet_Mordred_2D3D_from3Dmol.csv")

os.makedirs(out_root, exist_ok=True)

# =========================
# 2. Utility: build 3D mol from SMILES
# =========================

def mol_from_smiles_3d(smi: str):
    """
    SMILES -> RDKit Mol with 3D coordinates.
    Return None if embedding/optimization fails.
    """
    if not isinstance(smi, str):
        return None
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    try:
        mol = Chem.AddHs(mol)
        # Embed 3D conformer
        res = AllChem.EmbedMolecule(mol, randomSeed=42)
        if res != 0:
            return None
        # UFF optimize
        res_opt = AllChem.UFFOptimizeMolecule(mol, maxIters=200)
        # Buang H eksplisit kalau tidak ingin (opsional)
        mol = Chem.RemoveHs(mol)
        return mol
    except Exception:
        return None

# =========================
# 3. Setup Mordred Calculator (2D+3D)
# =========================
calc = Calculator(descriptors, ignore_3D=False)

# =========================
# 4. Proses hERG DevSet
# =========================

print("🔄 hERG - DevSet")
if not os.path.exists(file_path):
    print(f"[PERINGATAN] File tidak ditemukan: {file_path}")
    exit()

df = pd.read_csv(file_path)
assert "SMILES" in df.columns, f"Kolom 'SMILES' tidak ditemukan di {file_path}"
assert "Activity" in df.columns, f"Kolom 'Activity' tidak ditemukan di {file_path}"

smiles_list = df["SMILES"].astype(str).tolist()

# Bangun Mol 3D per SMILES
mols_3d = []
valid_idx = []
n_invalid = 0
for i, smi in enumerate(smiles_list):
    m = mol_from_smiles_3d(smi)
    if m is None:
        n_invalid += 1
    else:
        mols_3d.append(m)
        valid_idx.append(i)

if len(mols_3d) == 0:
    print("  [PERINGATAN] Tidak ada molekul 3D valid.")
    exit()

print(f"  Molekul input  : {len(smiles_list)}")
print(f"  Molekul 3D ok  : {len(mols_3d)}")
print(f"  Invalid (3D)   : {n_invalid}")

# Subset df ke hanya yang 3D-nya sukses
df_valid = df.iloc[valid_idx].reset_index(drop=True)

# Hitung Mordred di mols_3d
df_desc = calc.pandas(mols_3d)
df_desc = df_desc.dropna(axis=1, how="all").fillna(0)

# Gabungkan dengan SMILES + Activity (dan kolom lain jika ingin)
base_cols = ["SMILES", "Activity"]
other_cols = [c for c in df_valid.columns if c not in base_cols]
df_base = df_valid[base_cols + other_cols].reset_index(drop=True)

df_out = pd.concat([df_base, df_desc.reset_index(drop=True)], axis=1)

# Simpan
df_out.to_csv(out_path, index=False)

print(f"  Molekul tersimpan   : {len(df_out)}")
print(f"  Kolom descriptor    : {df_desc.shape[1]}")
print(f"  ✅ Saved: {out_path}")

# Preview metadata top 3 baris
print("  --- Preview top 3 ---")
preview_cols = ["SMILES", "Activity"] + list(df_desc.columns[:5])
print(df_out[preview_cols].head(3).to_string(index=False))
print("  --- End preview ---")

print("\n🎉 Perhitungan Mordred 2D+3D (hERG DevSet dengan RDKit 3D embedding) selesai!")


In [ ]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from mordred import Calculator, descriptors  # pip install "mordred[full]"

# =========================
# 1. Konfigurasi input & output (Nav1.5 saja)
# =========================

file_map = {
    "Nav1.5": {
        "DevSet":              r"D:\\QSAR Cardiotoxicity\\Dataset Splitting\\Nav1.5\\Nav1.5_DevSet.csv",
        "ExternalVal1_easy":   r"D:\\QSAR Cardiotoxicity\\Dataset Splitting\\Nav1.5\\Nav1.5_ExternalVal1_easy.csv",
        "ExternalVal2_medium": r"D:\\QSAR Cardiotoxicity\\Dataset Splitting\\Nav1.5\\Nav1.5_ExternalVal2_medium.csv",
        "ExternalVal3_hard":   r"D:\\QSAR Cardiotoxicity\\Dataset Splitting\\Nav1.5\\Nav1.5_ExternalVal3_hard.csv",
    }
}

out_root = r"D:\\QSAR Cardiotoxicity\\Descriptor Calculation\\Mordred_3D"
os.makedirs(out_root, exist_ok=True)

# =========================
# 2. Utility: build 3D mol from SMILES (NO UFF)
# =========================

def mol_from_smiles_3d_no_uff(smi: str):
    """
    SMILES -> RDKit Mol with 3D coordinates.
    Menggunakan ETKDG + fallback random, TANPA UFF/MMFF.
    Menghindari error 'UFFTYPER: Unrecognized atom type: S_6+6'.
    """
    if not isinstance(smi, str):
        return None
    smi = smi.strip()
    if not smi:
        return None

    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None

    try:
        # Tambah H
        mol = Chem.AddHs(mol)

        # Parameter ETKDG (direkomendasikan RDKit untuk 3D descriptor use-case)
        params = AllChem.ETKDG()
        params.randomSeed = 42
        # Kalau mau lebih robust:
        # params.useRandomCoords = True

        res = AllChem.EmbedMolecule(mol, params)
        if res != 0:
            # Fallback: random coords
            mol = Chem.AddHs(Chem.MolFromSmiles(smi))
            res2 = AllChem.EmbedMolecule(
                mol,
                useRandomCoords=True,
                randomSeed=123
            )
            if res2 != 0:
                return None

        # PENTING: tidak ada UFF/MMFF di sini
        # AllChem.UFFOptimizeMolecule(mol)  # JANGAN DIPAKAI
        # AllChem.MMFFOptimizeMolecule(mol) # JANGAN DIPAKAI

        mol = Chem.RemoveHs(mol)
        return mol
    except Exception:
        return None

# =========================
# 3. Setup Mordred Calculator (2D+3D)
# =========================
calc = Calculator(descriptors, ignore_3D=False)

# =========================
# 4. Loop Nav1.5: hitung Mordred + simpan SMILES & Activity
# =========================

for target, splits in file_map.items():
    for split_name, input_path in splits.items():
        print(f"\n🔄 {target} - {split_name}")
        if not os.path.exists(input_path):
            print(f"[PERINGATAN] File tidak ditemukan: {input_path}")
            continue

        df = pd.read_csv(input_path)
        assert "SMILES" in df.columns,   f"Kolom 'SMILES' tidak ditemukan di {input_path}"
        assert "Activity" in df.columns, f"Kolom 'Activity' tidak ditemukan di {input_path}"

        smiles_list = df["SMILES"].astype(str).tolist()
        print(f"  Total senyawa : {len(smiles_list)}")

        mols_3d = []
        valid_idx = []
        n_invalid = 0

        for i, smi in enumerate(smiles_list):
            if i % 200 == 0 and i > 0:
                print(f"    Progress: {i}/{len(smiles_list)} ({100*i/len(smiles_list):.0f}%)")

            m = mol_from_smiles_3d_no_uff(smi)
            if m is None:
                n_invalid += 1
            else:
                mols_3d.append(m)
                valid_idx.append(i)

        if len(mols_3d) == 0:
            print("  [PERINGATAN] Tidak ada molekul 3D valid, skip file ini.")
            continue

        print(f"  Molekul 3D ok : {len(mols_3d)}")
        print(f"  Invalid (3D)  : {n_invalid}")
        print(f"  Success rate  : {len(mols_3d)/len(smiles_list)*100:.1f}%")

        # Subset df ke hanya yang 3D-nya sukses
        df_valid = df.iloc[valid_idx].reset_index(drop=True)

        # Hitung Mordred di mols_3d
        print("  ⏳ Hitung Mordred descriptors...")
        df_desc = calc.pandas(mols_3d)
        df_desc = df_desc.dropna(axis=1, how="all").fillna(0)

        # Gabungkan dengan SMILES + Activity (dan kolom lain jika ingin)
        base_cols = ["SMILES", "Activity"]
        other_cols = [c for c in df_valid.columns if c not in base_cols]
        df_base = df_valid[base_cols + other_cols].reset_index(drop=True)

        df_out = pd.concat([df_base, df_desc.reset_index(drop=True)], axis=1)

        # Simpan
        out_name = f"{target}_{split_name}_Mordred_2D3D.csv"
        out_path = os.path.join(out_root, out_name)
        df_out.to_csv(out_path, index=False)

        print(f"  Molekul tersimpan   : {len(df_out)}")
        print(f"  Kolom descriptor    : {df_desc.shape[1]}")
        print(f"  ✅ Saved: {out_path}")

print("\n🎉 Perhitungan Mordred 2D+3D Nav1.5 selesai TANPA UFF/S_6+6 warning.")

# Mordred Reduction

In [ ]:
import os
import numpy as np
import pandas as pd

# =========================
# KONFIGURASI
# =========================
MORDRED_DIR = r"D:\QSAR Cardiotoxicity\Descriptor Calculation\Mordred_3D"
OUT_ROOT    = r"D:\QSAR Cardiotoxicity\Descriptor Calculation\Mordred_3D_reduced"
os.makedirs(OUT_ROOT, exist_ok=True)

# Meta columns yang harus dipertahankan
META_COLS = [
    "SMILES",
    "Activity",
    "pIC50",
    "Source",
    "Activity_Label",
    "Compound_ID",
    "InChIKey",
    "Max_Tanimoto_to_Dev",
]

CORR_TH = 0.95

DEV_FILES = {
    "Cav1.2": os.path.join(MORDRED_DIR, "Cav1.2_DevSet_Mordred_2D3D_from3Dmol.csv"),
    "hERG":   os.path.join(MORDRED_DIR, "hERG_DevSet_Mordred_2D3D_from3Dmol.csv"),
    "Nav1.5": os.path.join(MORDRED_DIR, "Nav1.5_DevSet_Mordred_2D3D.csv"),
}

for target, IN_PATH in DEV_FILES.items():
    print(f"\n=== Reducing Mordred DevSet: {target} ===")
    if not os.path.exists(IN_PATH):
        print(f"[WARNING] File not found: {IN_PATH}")
        continue

    df = pd.read_csv(IN_PATH)

    # 1) pisahkan meta vs descriptor
    meta_cols_present = [c for c in META_COLS if c in df.columns]
    desc_cols = [c for c in df.columns if c not in meta_cols_present]

    X = df[desc_cols].copy()

    # 2) paksa numerik (non-numeric -> NaN)
    X = X.apply(pd.to_numeric, errors="coerce")

    # 3) drop kolom all-NaN
    all_nan_cols = X.columns[X.isna().all()].tolist()
    X = X.drop(columns=all_nan_cols)

    # 4) imputasi mean
    X = X.fillna(X.mean(numeric_only=True))

    # 5) drop zero-variance
    var = X.var(axis=0)
    zero_var_cols = var[var == 0].index.tolist()
    X = X.drop(columns=zero_var_cols)

    # 6) drop fitur highly correlated (|r| > CORR_TH)
    corr = X.corr(method="pearson").abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > CORR_TH)]
    X_red = X.drop(columns=to_drop)

    # 7) gabungkan lagi + simpan
    df_out = pd.concat(
        [df[meta_cols_present].reset_index(drop=True),
         X_red.reset_index(drop=True)],
        axis=1
    )

    base = os.path.splitext(os.path.basename(IN_PATH))[0]
    OUT_PATH = os.path.join(OUT_ROOT, f"{base}_reduced_r095.csv")
    df_out.to_csv(OUT_PATH, index=False)

    print("Rows:", len(df))
    print("Meta cols:", len(meta_cols_present))
    print("Mordred awal:", len(desc_cols))
    print("Drop all-NaN cols:", len(all_nan_cols))
    print("Drop zero-variance cols:", len(zero_var_cols))
    print("Drop correlated cols (>|r|>{}):".format(CORR_TH), len(to_drop))
    print("Mordred akhir:", X_red.shape[1])
    print("Saved DevSet reduced:", OUT_PATH)

    # simpan daftar kolom descriptor yang dipakai
    KEPT_COLS_CSV = OUT_PATH.replace(".csv", "_kept_columns.csv")
    pd.DataFrame({"column": list(X_red.columns)}).to_csv(KEPT_COLS_CSV, index=False)
    print("Kept descriptor list:", KEPT_COLS_CSV)


# Mordred Reduction

In [ ]:
import numpy as np
import pandas as pd

# ====== PATH (EDIT) ======
IN_PATH  = r"D:\QSAR Cardiotoxicity\Nav1.5\dev\Nav1.5_development_3D_mordred_full_with_labels.csv"
OUT_PATH = r"D:\QSAR Cardiotoxicity\Nav1.5\dev\hERG_development_3D_mordred_reduced_r095.csv"

META_COLS = [
    "target", "split", "dataset_name", "mol_index", "mol_id",
    "SMILES", "pIC50", "Source", "Outcome", "Activity_Label"
]
CORR_TH = 0.95
# =========================

df = pd.read_csv(IN_PATH)

# 1) Pisahkan metadata vs mordred
meta_cols = [c for c in META_COLS if c in df.columns]
mordred_cols = [c for c in df.columns if c not in meta_cols]

X = df[mordred_cols].copy()

# 2) Paksa numerik; string/non-numerik jadi NaN
X = X.apply(pd.to_numeric, errors="coerce")

# 3) Drop kolom yang semuanya NaN (tidak pernah terhitung)
all_nan_cols = X.columns[X.isna().all()].tolist()
X = X.drop(columns=all_nan_cols)

# 4) Imputasi mean untuk missing
X = X.fillna(X.mean(numeric_only=True))

# 5) Drop zero-variance (konstan)
var = X.var(axis=0)
zero_var_cols = var[var == 0].index.tolist()
X = X.drop(columns=zero_var_cols)

# 6) Drop fitur yang highly correlated (|r| > 0.95)
#    (ambil upper triangle supaya tidak dobel)
corr = X.corr(method="pearson").abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > CORR_TH)]
X_red = X.drop(columns=to_drop)

# 7) Gabungkan lagi + simpan
df_out = pd.concat([df[meta_cols], X_red], axis=1)
df_out.to_csv(OUT_PATH, index=False)

# 8) Report ringkas
print("=== Mordred reduction report ===")
print("Rows:", len(df))
print("Mordred awal:", len(mordred_cols))
print("Drop all-NaN cols:", len(all_nan_cols))
print("Drop zero-variance cols:", len(zero_var_cols))
print("Drop correlated cols (>|r|>{}):".format(CORR_TH), len(to_drop))
print("Mordred akhir:", X_red.shape[1])
print("Saved:", OUT_PATH)

# ====== (TAMBAHAN) OUTPUT LIST KOLUM ======
OUT_DROPPED = OUT_PATH.replace(".csv", "_dropped_columns.csv")
OUT_KEPT    = OUT_PATH.replace(".csv", "_kept_columns.csv")

dropped_report = pd.DataFrame({
    "column": (all_nan_cols + zero_var_cols + to_drop),
    "reason": (["all_nan"] * len(all_nan_cols)) +
              (["zero_variance"] * len(zero_var_cols)) +
              ([f"corr_gt_{CORR_TH}"] * len(to_drop))
})

# (opsional) hilangkan duplikat kalau ada kolom masuk 2 kategori (jarang, tapi aman)
dropped_report = dropped_report.drop_duplicates(subset=["column"], keep="first")

kept_cols = list(X_red.columns)
kept_report = pd.DataFrame({"column": kept_cols})

dropped_report.to_csv(OUT_DROPPED, index=False)
kept_report.to_csv(OUT_KEPT, index=False)

print("\n=== Column reports saved ===")
print("Dropped columns file:", OUT_DROPPED)
print("Kept columns file   :", OUT_KEPT)

# (opsional) preview beberapa kolom
print("\nDropped preview (first 20):")
print(dropped_report.head(20).to_string(index=False))

print("\nKept preview (first 20):")
print(kept_report.head(20).to_string(index=False))
# =========================================


# Standarisasi Mordred

In [ ]:
import json
import numpy as np
import pandas as pd

# ================== EDIT PATH DI SINI ==================
DATASETS = {
    "Nav1.5": r"D:\QSAR Cardiotoxicity\Nav1.5\dev\Nav1.5_development_3D_mordred_reduced_r095.csv",
    "hERG":   r"D:\QSAR Cardiotoxicity\hERG\dev\hERG_development_3D_mordred_reduced_r095.csv",
    "Cav1.2": r"D:\QSAR Cardiotoxicity\Cav1.2\dev set\Cav1.2_development_3D_mordred_reduced_r095.csv",
}

# folder output (boleh kamu ganti)
OUT_DIR = r"D:\QSAR Cardiotoxicity\_mordred_ready"

META_COLS = [
    "target", "split", "dataset_name", "mol_index", "mol_id",
    "SMILES", "pIC50", "Source", "Outcome", "Activity_Label"
]
# =======================================================


def ensure_dir(path):
    import os
    os.makedirs(path, exist_ok=True)


def split_meta_mordred(df, meta_cols):
    meta_cols = [c for c in meta_cols if c in df.columns]
    mordred_cols = [c for c in df.columns if c not in meta_cols]
    return meta_cols, mordred_cols


def fit_preprocess_train(df_train, meta_cols):
    """Fit imputer mean + standard scaler on train (development set)."""
    meta_cols_present, mordred_cols = split_meta_mordred(df_train, meta_cols)

    X = df_train[mordred_cols].copy()
    X = X.apply(pd.to_numeric, errors="coerce")

    # (1) buang kolom all-NaN (kalau masih ada)
    all_nan_cols = X.columns[X.isna().all()].tolist()
    X = X.drop(columns=all_nan_cols)

    # (2) imputasi mean (fit di train)
    means = X.mean(axis=0, numeric_only=True)
    X = X.fillna(means)

    # (3) standardisasi (fit di train): mean=0, var=1
    mu = X.mean(axis=0)
    sd = X.std(axis=0, ddof=0)

    # kolom dengan std=0 (jarang karena sudah reduced, tapi aman)
    nonzero = sd[sd > 0].index.tolist()
    X = X[nonzero]
    mu = mu[nonzero]
    sd = sd[nonzero]

    Xs = (X - mu) / sd

    params = {
        "meta_cols": meta_cols_present,
        "mordred_cols_used": nonzero,
        "dropped_all_nan_cols": all_nan_cols,
        "imputer_means": means.to_dict(),
        "scaler_mean": mu.to_dict(),
        "scaler_std": sd.to_dict(),
    }

    df_out = pd.concat([df_train[meta_cols_present], Xs], axis=1)
    return df_out, params


def transform_with_params(df, params):
    """Apply train-fitted imputer+scaler to any dataset (val/test/external)."""
    meta_cols = params["meta_cols"]
    cols = params["mordred_cols_used"]

    # reindex supaya kolom urut & konsisten; yang hilang jadi NaN
    X = df.reindex(columns=cols).copy()
    X = X.apply(pd.to_numeric, errors="coerce")

    means = pd.Series(params["imputer_means"]).reindex(cols)
    mu    = pd.Series(params["scaler_mean"]).reindex(cols)
    sd    = pd.Series(params["scaler_std"]).reindex(cols)

    X = X.fillna(means)
    Xs = (X - mu) / sd

    df_out = pd.concat([df[meta_cols], Xs], axis=1)
    return df_out


# ================== RUN ==================
ensure_dir(OUT_DIR)

for target, path in DATASETS.items():
    df = pd.read_csv(path)

    # Arab et al: preprocessing dilakukan sebagai pipeline lalu dipakai untuk model.[file:88]
    df_train_ready, params = fit_preprocess_train(df, META_COLS)

    out_train = rf"{OUT_DIR}\{target}_mlp_ready_train_scaled.csv"
    out_json  = rf"{OUT_DIR}\{target}_scaler_params.json"

    df_train_ready.to_csv(out_train, index=False)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(params, f, indent=2)

    print(f"\n=== {target} ===")
    print("Input:", path)
    print("Saved train-ready:", out_train)
    print("Saved params:", out_json)
    print("Rows:", len(df_train_ready))
    print("Mordred cols used:", len(params["mordred_cols_used"]))
    print("Dropped all-NaN cols:", len(params["dropped_all_nan_cols"]))
